<a href="https://colab.research.google.com/github/joyashre/LLM-NLP-Learning-task/blob/main/NLP_Task_7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 7

In [ ]:
# Cell 1: Install required libraries
!pip install -q transformers datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.3 MB/s eta 0:00:00


In [ ]:
# Cell 2: Create Dataset and Formatting Prompts
import pandas as pd
from datasets import Dataset

# 1. Construct a small dataset for Proof of Concept (PoC)
mock_data = [
    # Train Data
    {"context": "I deposited all my cash into the bank today.", "word": "bank", "sense": "financial institution"},
    {"context": "We sat by the river bank and watched the sunset.", "word": "bank", "sense": "sloping land beside water"},
    {"context": "The cricket player swung his bat with full force.", "word": "bat", "sense": "wooden club used in sports"},
    {"context": "A scary bat flew out of the dark cave at night.", "word": "bat", "sense": "flying mammal"},
    {"context": "Please bank the airplane to the left.", "word": "bank", "sense": "tilt or turn"},
    {"context": "He bought a new baseball bat.", "word": "bat", "sense": "wooden club used in sports"},

    # Dev Data
    {"context": "The bank approved my home loan.", "word": "bank", "sense": "financial institution"},
    {"context": "The boat crashed into the muddy bank.", "word": "bank", "sense": "sloping land beside water"},

    # Test Data
    {"context": "The vampire turned into a bat and flew away.", "word": "bat", "sense": "flying mammal"},
    {"context": "I need to visit the bank to withdraw money.", "word": "bank", "sense": "financial institution"}
]

In [ ]:
df = pd.DataFrame(mock_data)

# 2. Prompt Dataset Building (Format for Causal LM)
def format_prompt(row):
    return f"Context: {row['context']}\nQuestion: What is the meaning of the word '{row['word']}' in this context?\nAnswer: {row['sense']}<|endoftext|>"

df['text'] = df.apply(format_prompt, axis=1)

df

,context,word,sense,text
0,I deposited all my cash into the bank today.,bank,financial institution,Context: I deposited all my cash into the bank...
1,We sat by the river bank and watched the sunset.,bank,sloping land beside water,Context: We sat by the river bank and watched ...
2,The cricket player swung his bat with full force.,bat,wooden club used in sports,Context: The cricket player swung his bat with...
3,A scary bat flew out of the dark cave at night.,bat,flying mammal,Context: A scary bat flew out of the dark cave...
4,Please bank the airplane to the left.,bank,tilt or turn,Context: Please bank the airplane to the left....
5,He bought a new baseball bat.,bat,wooden club used in sports,Context: He bought a new baseball bat.\nQuesti...
6,The bank approved my home loan.,bank,financial institution,Context: The bank approved my home loan.\nQues...
7,The boat crashed into the muddy bank.,bank,sloping land beside water,Context: The boat crashed into the muddy bank....
8,The vampire turned into a bat and flew away.,bat,flying mammal,Context: The vampire turned into a bat and fle...
9,I need to visit the bank to withdraw money.,bank,financial institution,Context: I need to visit the bank to withdraw ...


In [ ]:
# 3. Train, Dev, Test Splits
train_df = df.iloc[:6]
dev_df = df.iloc[6:8]
test_df = df.iloc[8:]

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

print("--- Example Formatted Prompt ---")
print(train_dataset['text'][0])

--- Example Formatted Prompt ---
Context: I deposited all my cash into the bank today.
Question: What is the meaning of the word 'bank' in this context?
Answer: financial institution<|endoftext|>


Setup Model and Tokenizer

In [ ]:
# Cell 3: Load Model and Tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "distilgpt2"

print(f"Loading tokenizer and model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# GPT-2 doesnt have a padding token by default, so we add one
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto" # Automatically uses GPU if available
)
print("Model loaded successfully!")

Loading tokenizer and model: distilgpt2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


Finetuning Model using SFTTrainer



In [ ]:
# Cell 4: Finetuning Model using SFTTrainer
from trl import SFTTrainer, SFTConfig

# Naye update mein parameters ke naam badal gaye hain
training_args = SFTConfig(
    output_dir="./wsd_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=2,
    learning_rate=5e-5,
    save_strategy="epoch",
    dataset_text_field="text",
    max_length=128               # <-- Yahan 'max_seq_length' ki jagah sirf 'max_length' aayega
)

# Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    args=training_args,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

# Save final model checkpoint
final_model_path = "./wsd_final_model"
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\nModel saved successfully at {final_model_path}")

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully at ./wsd_final_model


Load Saved Checkpoint and Perform Inference

In [ ]:
# Cell 5: Load Checkpoint, Perform Inference and Evaluate
from transformers import pipeline

print("Loading saved model for inference...")
# Load the model we just saved
finetuned_model = AutoModelForCausalLM.from_pretrained(final_model_path)
finetuned_tokenizer = AutoTokenizer.from_pretrained(final_model_path)

# Create a text-generation pipeline
wsd_pipeline = pipeline(
    "text-generation",
    model=finetuned_model,
    tokenizer=finetuned_tokenizer,
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available
)

print("\n--- Inference on Test Set ---")
for i, row in test_df.iterrows():
    # We only pass the context and question to the model (not the answer)
    prompt = f"Context: {row['context']}\nQuestion: What is the meaning of the word '{row['word']}' in this context?\nAnswer:"

    # Generate the output
    outputs = wsd_pipeline(
        prompt,
        max_new_tokens=10, # Give it space to write the meaning
        num_return_sequences=1,
        pad_token_id=finetuned_tokenizer.eos_token_id,
        temperature=0.1 # Low temperature for more deterministic/factual output
    )

    generated_text = outputs[0]['generated_text']

    print(f"Test Context: {row['context']}")
    print(f"Target Word: '{row['word']}'")
    print(f"Expected Sense: {row['sense']}")
    # Extract only the generated answer part
    print(f"Model Output: {generated_text[len(prompt):].strip()}\n")
    print("-" * 50)

Loading saved model for inference...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Inference on Test Set ---


Both `max_new_tokens` (=10) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Context: The vampire turned into a bat and flew away.
Target Word: 'bat'
Expected Sense: flying mammal
Model Output: bat
Question: bat
Question: bat

--------------------------------------------------
Test Context: I need to visit the bank to withdraw money.
Target Word: 'bank'
Expected Sense: financial institution
Model Output: bank
Question: bank
Question: bank

--------------------------------------------------
